## imports

In [24]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, BatchNormalization, Input,
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import set_random_seed

SEED = 42
np.random.seed(SEED)
set_random_seed(SEED)

## reading and preparing data

In [25]:
df0404 = pd.read_csv("../data/labeledAndAnomaly/no_plug/0404.csv")
df0712 = pd.read_csv("../data/labeledAndAnomaly/no_plug/0712.csv")
df1703 = pd.read_csv("../data/labeledAndAnomaly/no_plug/1703.csv")
df2906_1 = pd.read_csv("../data/labeledAndAnomaly/no_plug/2906-1.csv")
df3006_1 = pd.read_csv("../data/labeledAndAnomaly/no_plug/3006-1.csv")
df3006_2 = pd.read_csv("../data/labeledAndAnomaly/no_plug/3006-2.csv")

df1103 = pd.read_csv("../data/labeledAndAnomaly/partial_plug/1103.csv")
df1803 = pd.read_csv("../data/labeledAndAnomaly/partial_plug/1803.csv")
df2503 = pd.read_csv("../data/labeledAndAnomaly/partial_plug/2503.csv")


df0612_2 = pd.read_csv("../data/labeledAndAnomaly/plug/0612-2.csv")
df0612_3 = pd.read_csv("../data/labeledAndAnomaly/plug/0612-3.csv")
df0707_1 = pd.read_csv("../data/labeledAndAnomaly/plug/0707-1.csv")
df0707_2 = pd.read_csv("../data/labeledAndAnomaly/plug/0707-2.csv")
df0707_4 = pd.read_csv("../data/labeledAndAnomaly/plug/0707-4.csv")
df0709 = pd.read_csv("../data/labeledAndAnomaly/plug/0709.csv")
df1003 = pd.read_csv("../data/labeledAndAnomaly/plug/1003.csv")
df1112 = pd.read_csv("../data/labeledAndAnomaly/plug/1112.csv")
df1310_2 = pd.read_csv("../data/labeledAndAnomaly/plug/1310-2.csv")
df1310_3 = pd.read_csv("../data/labeledAndAnomaly/plug/1310-3.csv")
df1407_1 = pd.read_csv("../data/labeledAndAnomaly/plug/1407-1.csv")
df1407_2 = pd.read_csv("../data/labeledAndAnomaly/plug/1407-2.csv")
df1407_3 = pd.read_csv("../data/labeledAndAnomaly/plug/1407-3.csv")
df1407_4 = pd.read_csv("../data/labeledAndAnomaly/plug/1407-4.csv")
df1503 = pd.read_csv("../data/labeledAndAnomaly/plug/1503.csv")
df1811 = pd.read_csv("../data/labeledAndAnomaly/plug/1811.csv")
df2106_2 = pd.read_csv("../data/labeledAndAnomaly/plug/2106-2.csv")
df2108_1 = pd.read_csv("../data/labeledAndAnomaly/plug/2108-1.csv")
df2108_2 = pd.read_csv("../data/labeledAndAnomaly/plug/2108-2.csv")
df2701 = pd.read_csv("../data/labeledAndAnomaly/plug/2701.csv")
df2906_2 = pd.read_csv("../data/labeledAndAnomaly/plug/2906-2.csv")
df2906_3 = pd.read_csv("../data/labeledAndAnomaly/plug/2906-3.csv")
df2906_4 = pd.read_csv("../data/labeledAndAnomaly/plug/2906-4.csv")
df3006_3 = pd.read_csv("../data/labeledAndAnomaly/plug/3006-3.csv")

In [26]:
no_plug_dfs = [df0404, df0712, df1703, df2906_1, df3006_1, df3006_2, df1103, df1803, df2503]
plug_dfs =  [df0612_2, df0612_3, df0707_1, df0707_2, df0707_4, df0709, df1003, df1112, df1310_2, df1310_3,
             df1407_1, df1407_2, df1407_3, df1407_4, df1503, df1811, df2106_2, df2108_1, df2108_2, df2701, df2906_2, df2906_3, df2906_4, df3006_3]

In [27]:
for df in no_plug_dfs + plug_dfs:
    df = df.dropna().reset_index(drop=True)

In [28]:
label = "Plug_Label"

features = [
    "TS outlet pressure (Mean)",
    "TS inlet pressure (Mean)",
    "Pump outlet pressure (Mean)",
    "Temperature TS outlet (Mean)",
    "Tank temperature (Mean)",
    "Temperature TS inlet (Mean)",
    "Bypass temperature (Mean)",
    "Differential pressure (Mean)",
    "Total system DP",
    "Pump to inlet DP",
    "Relative_DP",
    "Pressure_Ratio",
    "Relative_Ratio_Dev",
    "rel_tot_dp_change",
    "rel_ts_dp_change",
    "rel_p_ts_dp_change",
    "anomaly_score",
]

In [ ]:
window = 120 
horizon     = 60  

LSTM_UNITS_1  = 64
LSTM_UNITS_2  = 32
DENSE_UNITS   = 16
DROPOUT_1     = 0.3
DROPOUT_2     = 0.2
LEARNING_RATE = 1e-3
BATCH_SIZE    = 64
MAX_EPOCHS    = 150

N_FOLDS = 5

OUTPUT_DIR = Path("../outputs")

In [30]:
def make_sequences(df: pd.DataFrame,
                   window: int = window,
                   horizon: int = horizon):

    data   = df[features].values.astype(np.float32)
    labels = df[label].values.astype(np.int32)
    X, y = [], []
    max_i = len(df) - window - horizon


    for i in range(max_i):
        X.append(data[i : i + window])
        future = labels[i + window : i + window + horizon]
        y.append(int(future.max()))

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

In [ ]:
def assemble_dataset(dfs, run_id_offset = 0):
    X_parts, y_parts, g_parts = [], [], []

    for i, dfs in enumerate(dfs):
        Xi, yi = make_sequences(df)

        gi = np.full(len(yi), i + run_id_offset, dtype=np.int32)
        X_parts.append(Xi)
        y_parts.append(yi)
        g_parts.append(gi)

    X = np.concatenate(X_parts, axis=0)
    y = np.concatenate(y_parts, axis=0)
    g = np.concatenate(g_parts, axis=0)
    return X, y, g

In [32]:
def fit_scale(X_train_3d, X_val_3d):
    """
    Flatten to 2-D, fit StandardScaler on train, transform both.
    Returns scaled 3-D arrays.
    """
    n_tr, w, f = X_train_3d.shape
    n_va        = X_val_3d.shape[0]

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(
        X_train_3d.reshape(-1, f)
    ).reshape(n_tr, w, f)

    X_va_scaled = scaler.transform(
        X_val_3d.reshape(-1, f)
    ).reshape(n_va, w, f)

    return X_tr_scaled, X_va_scaled, scaler

In [33]:
def build_model(window: int, n_features: int) -> tf.keras.Model:
    model = Sequential([
        Input(shape=(window, n_features)),

        LSTM(LSTM_UNITS_1, return_sequences=True),
        BatchNormalization(),
        Dropout(DROPOUT_1),

        LSTM(LSTM_UNITS_2, return_sequences=False),
        BatchNormalization(),
        Dropout(DROPOUT_2),

        Dense(DENSE_UNITS, activation="relu"),
        Dense(3, activation="softmax"),
    ], name="plug_lstm")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [34]:
def get_callbacks():
    return [
        EarlyStopping(
            monitor="val_loss",
            patience=12,
            restore_best_weights=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=6,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

In [35]:
def evaluate_fold(model, X_val, y_val, fold_idx):
    """Print and return metrics for one fold."""
    y_pred_proba = model.predict(X_val, verbose=0)
    y_pred       = y_pred_proba.argmax(axis=1)

    report = classification_report(
        y_val, y_pred,
        target_names=["normal", "warning", "critical"],
        output_dict=True,
        zero_division=0,
    )
    macro_f1 = f1_score(y_val, y_pred, average="macro", zero_division=0)

    print(f"\n  Fold {fold_idx + 1} results")
    print(classification_report(
        y_val, y_pred,
        target_names=["normal", "warning", "critical"],
        zero_division=0,
    ))

    cm = confusion_matrix(y_val, y_pred, labels=[0, 1, 2])
    plot_confusion_matrix(cm, fold_idx)

    return {
        "fold":      fold_idx,
        "macro_f1":  macro_f1,
        "report":    report,
        "y_val":     y_val,
        "y_pred":    y_pred,
        "y_proba":   y_pred_proba,
    }

## plot functions

In [36]:
def plot_confusion_matrix(cm, fold_idx):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", ax=ax,
        xticklabels=["normal", "warning", "critical"],
        yticklabels=["normal", "warning", "critical"],
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"Confusion matrix — fold {fold_idx + 1}")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"confusion_fold_{fold_idx + 1}.png", dpi=120)
    plt.close()

In [37]:
def plot_training_history(history, fold_idx):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].plot(history.history["loss"],     label="train loss")
    axes[0].plot(history.history["val_loss"], label="val loss")
    axes[0].set_title(f"Loss — fold {fold_idx + 1}")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history.history["accuracy"],     label="train acc")
    axes[1].plot(history.history["val_accuracy"], label="val acc")
    axes[1].set_title(f"Accuracy — fold {fold_idx + 1}")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"history_fold_{fold_idx + 1}.png", dpi=120)
    plt.close()

In [38]:
def plot_prediction_timeline(results, fold_idx):
    """
    For the validation windows in this fold, plot the true label and
    predicted class (and warning probability) over time.
    Useful for checking how many steps ahead the warning fires.
    """
    y_val   = results["y_val"]
    y_pred  = results["y_pred"]
    y_proba = results["y_proba"]

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

    axes[0].plot(y_val,  label="True label",      linewidth=1.2)
    axes[0].plot(y_pred, label="Predicted label",  linewidth=1.2, alpha=0.8)
    axes[0].set_ylabel("Class")
    axes[0].set_yticks([0, 1, 2])
    axes[0].set_yticklabels(["normal", "warning", "critical"])
    axes[0].legend(loc="upper left")
    axes[0].set_title(f"Prediction timeline — fold {fold_idx + 1}")

    axes[1].stackplot(
        range(len(y_proba)),
        y_proba[:, 0], y_proba[:, 1], y_proba[:, 2],
        labels=["P(normal)", "P(warning)", "P(critical)"],
        colors=["steelblue", "orange", "crimson"],
        alpha=0.75,
    )
    axes[1].set_ylabel("Probability")
    axes[1].set_xlabel("Window index")
    axes[1].legend(loc="upper left")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"timeline_fold_{fold_idx + 1}.png", dpi=120)
    plt.close()


## main

In [51]:
def main():

    X_clean, y_clean, _ = assemble_dataset(no_plug_dfs, run_id_offset=0)

    X_plug, y_plug, g_plug = assemble_dataset(plug_dfs, run_id_offset=len(no_plug_dfs)
    )

    n_features = X_plug.shape[2]

    gkf      = GroupKFold(n_splits=N_FOLDS)
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(
        gkf.split(X_plug, y_plug, g_plug)):

        val_runs = np.unique(g_plug[val_idx]) - len(CLEAN_RUN_FILES)
        print(f"  Fold {fold_idx + 1}/{N_FOLDS}  |  val plug runs: {val_runs + 1}")


        X_train_raw = np.concatenate([X_clean, X_plug[train_idx]], axis=0)
        y_train     = np.concatenate([y_clean, y_plug[train_idx]], axis=0)
        X_val_raw = X_plug[val_idx]
        y_val     = y_plug[val_idx]

        X_train, X_val, _ = fit_scale(X_train_raw, X_val_raw)


        classes = np.unique(y_train)
        weights = compute_class_weight("balanced", classes=classes, y=y_train)
        cw      = dict(zip(classes.tolist(), weights.tolist()))

        model = build_model(window, n_features)
        if fold_idx == 0:
            model.summary()

        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=MAX_EPOCHS,
            batch_size=BATCH_SIZE,
            class_weight=cw,
            callbacks=get_callbacks(),
            verbose=1,
        )

        model.save(OUTPUT_DIR / f"model_fold_{fold_idx + 1}.keras")

        # Evaluate + plot
        plot_training_history(history, fold_idx)
        result = evaluate_fold(model, X_val, y_val, fold_idx)
        plot_prediction_timeline(result, fold_idx)
        fold_results.append(result)

In [50]:
if __name__ == "__main__":
    main()

  Fold 1/5  |  val plug runs: [ 5 10 15 20 24]


Model: "plug_lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_20 (LSTM)                  │ (None, 120, 64)        │        20,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (None, 120, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 120, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_21 (LSTM)                  │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_21          │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,371 (134.26 KB)

 Trainable params: 34,179 (133.51 KB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/150
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 50s 41ms/step - accuracy: 0.9882 - loss: 0.0438 - val_accuracy: 0.9993 - val_loss: 0.0016 - learning_rate: 0.0010
Epoch 2/150
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 48s 41ms/step - accuracy: 0.9996 - loss: 0.0014 - val_accuracy: 0.9996 - val_loss: 4.5417e-04 - learning_rate: 0.0010
Epoch 3/150
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 48s 41ms/step - accuracy: 0.9993 - loss: 0.0016 - val_accuracy: 1.0000 - val_loss: 2.1654e-04 - learning_rate: 0.0010
Epoch 4/150
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 49s 42ms/step - accuracy: 0.9966 - loss: 0.0127 - val_accuracy: 0.9996 - val_loss: 6.8847e-04 - learning_rate: 0.0010
Epoch 5/150
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 49s 41ms/step - accuracy: 0.9998 - loss: 7.3252e-04 - val_accuracy: 0.9996 - val_loss: 4.3062e-04 - learning_rate: 0.0010
Epoch 6/150
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 48s 41ms/step - accuracy: 0.9996 - loss: 8.4892e-04 - val_accuracy: 1.0000 - val_loss: 2.2480e-04 - learning_rate: 0.0010
Epoch 7/150
1173/1173 ━━━━━━━━

NameError: name 'X_va' is not defined

In [ ]:
print(f"\n[4/4] Cross-validation summary")
    print("=" * 55)
    macro_f1s = [r["macro_f1"] for r in fold_results]
    for i, f1 in enumerate(macro_f1s):
        print(f"  Fold {i+1}: macro F1 = {f1:.4f}")
    print(f"\n  Mean macro F1 : {np.mean(macro_f1s):.4f}")
    print(f"  Std  macro F1 : {np.std(macro_f1s):.4f}")

    # Per-class F1 summary
    for cls_name in ["normal", "warning", "critical"]:
        f1s = [r["report"][cls_name]["f1-score"] for r in fold_results]
        print(f"  {cls_name:10s} F1 — mean {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")

    # ── Retrain final model on ALL data ──────────────────────────────────────
    print("\nRetraining final model on all data...")
    X_all_raw = np.concatenate([X_clean, X_plug], axis=0)
    y_all     = np.concatenate([y_clean, y_plug],  axis=0)

    # For the final model, scale on the full dataset
    n, w, f   = X_all_raw.shape
    final_scaler = StandardScaler()
    X_all     = final_scaler.fit_transform(
        X_all_raw.reshape(-1, f)
    ).reshape(n, w, f)

    classes = np.unique(y_all)
    weights = compute_class_weight("balanced", classes=classes, y=y_all)
    cw_all  = dict(zip(classes.tolist(), weights.tolist()))

    final_model = build_model(WINDOW_SIZE, n_features)
    final_model.fit(
        X_all, y_all,
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=cw_all,
        callbacks=[
            # No val_loss here — use a fixed epoch count from best CV fold
            tf.keras.callbacks.EarlyStopping(
                monitor="loss", patience=15, restore_best_weights=True
            )
        ],
        verbose=1,
    )

    final_model.save(OUTPUT_DIR / "model_final.keras")

    # Save scaler for inference
    import pickle
    with open(OUTPUT_DIR / "scaler_final.pkl", "wb") as f_out:
        pickle.dump(final_scaler, f_out)

    print(f"\nDone. Outputs saved to: {OUTPUT_DIR.resolve()}")
    print("  model_final.keras   — deploy this for inference")
    print("  scaler_final.pkl    — apply this before inference")
    print("  model_fold_N.keras  — per-fold models for inspection")
    print("  confusion_fold_N.png / history_fold_N.png / timeline_fold_N.png")

In [ ]:
def predict_on_new_run(csv_path: str,
                       model_path: str = "outputs/model_final.keras",
                       scaler_path: str = "outputs/scaler_final.pkl"):
    """
    Run the trained model on a new CSV file and return a DataFrame with
    the predicted class and class probabilities at each window position.

    Usage
    -----
        results = predict_on_new_run("data/new_run.csv")
        results.to_csv("outputs/new_run_predictions.csv", index=False)
        print(results[results["pred_label"] >= 1])   # show all warnings
    """
    import pickle

    model  = tf.keras.models.load_model(model_path)
    with open(scaler_path, "rb") as f_in:
        scaler = pickle.load(f_in)

    df = load_csv(Path(csv_path))
    X, _ = make_sequences(df)   # labels from make_sequences not used here

    if len(X) == 0:
        print("Run too short to generate sequences.")
        return pd.DataFrame()

    n, w, f = X.shape
    X_scaled = scaler.transform(X.reshape(-1, f)).reshape(n, w, f)

    proba = model.predict(X_scaled, verbose=0)
    pred  = proba.argmax(axis=1)

    results = pd.DataFrame({
        "window_start":    range(len(pred)),
        "pred_label":      pred,
        "P_normal":        proba[:, 0],
        "P_warning":       proba[:, 1],
        "P_critical":      proba[:, 2],
    })

    # Map window index back to the approximate timestep in the original run
    results["approx_timestep"] = results["window_start"] + WINDOW_SIZE

    # Flag first window where model predicts warning or critical
    first_warning  = results[results["pred_label"] >= 1].index
    first_critical = results[results["pred_label"] == 2].index

    if len(first_warning) > 0:
        print(f"First WARNING  at window {first_warning[0]}  "
              f"(timestep ~{results.loc[first_warning[0], 'approx_timestep']})")
    if len(first_critical) > 0:
        print(f"First CRITICAL at window {first_critical[0]}  "
              f"(timestep ~{results.loc[first_critical[0], 'approx_timestep']})")

    return results